In [12]:
pip install openai datasets pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import re
import pandas as pd
from datasets import load_dataset
from openai import OpenAI

In [14]:
# 1. Setup LM Studio Client with the specified IP and Port
client = OpenAI(base_url="http://172.17.106.218:1234/v1", api_key="lm-studio")

In [15]:



# 2. Configuration
NUM_QUESTIONS = 400
SAVE_INTERVAL = 2
SAMPLES_FOR_UNCERTAINTY = 6
HALLUCINATION_THRESHOLD = 0.8
CSV_FILENAME = "offline_llm_medmcqa_results_LLama_med42.csv"

def get_loaded_model():
    """Queries LM Studio to check which model file is currently loaded."""
    try:
        print("Checking connection to LM Studio and loaded model...")
        models = client.models.list()
        if models.data:
            # LM Studio returns the filename or model identifier here
            model_id = models.data[0].id
            print(f"SUCCESS: Currently loaded model is -> {model_id}\n")
            return model_id
        else:
            print("WARNING: Connected to LM Studio, but no model appears to be loaded.")
            return None
    except Exception as e:
        print(f"ERROR: Could not connect to LM Studio at http://172.17.103.35:1234/v1. \nDetails: {e}")
        return None

def get_llm_response(prompt_text, temp=0.7, max_tokens=150):
    """Fetches a response from the local LLM via LM Studio."""
    try:
        response = client.chat.completions.create(
            model="local-model", # The API requires a string here, but LM Studio uses whatever is loaded
            messages=[
                {"role": "system", "content": "You are a highly accurate medical AI assistant."},
                {"role": "user", "content": prompt_text}
            ],
            temperature=temp,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error communicating with LM Studio: {e}")
        return ""

def extract_option(text):
    """Extracts the chosen option (A, B, C, or D) from the text."""
    match = re.search(r'\b([A-D])\b', text.upper())
    return match.group(1) if match else None

def evaluate_question(item):
    """Evaluates a single MedMCQA question for accuracy and hallucination."""
    question = item['question']
    options = {
        'A': item['opa'], 'B': item['opb'], 
        'C': item['opc'], 'D': item['opd']
    }
    
    # Ground truth mapping for MedMCQA (1=A, 2=B, 3=C, 4=D)
    gt_map = {1: 'A', 2: 'B', 3: 'C', 4: 'D'}
    correct_option = gt_map.get(item['cop'], 'A') 
    
    mcq_prompt = (
        f"Medical Question: {question}\n"
        f"A) {options['A']}\nB) {options['B']}\nC) {options['C']}\nD) {options['D']}\n\n"
        "Provide only the letter of the correct option (A, B, C, or D)."
    )

    # 1. Measure Semantic Uncertainty (Sampling multiple times)
    sampled_answers = []
    for _ in range(SAMPLES_FOR_UNCERTAINTY):
        response = get_llm_response(mcq_prompt, temp=0.8, max_tokens=10)
        opt = extract_option(response)
        if opt:
            sampled_answers.append(opt)

    # Calculate Semantic Score (Agreement ratio)
    if not sampled_answers:
        semantic_score = 0.0
        final_prediction = "None"
    else:
        most_common = max(set(sampled_answers), key=sampled_answers.count)
        semantic_score = sampled_answers.count(most_common) / len(sampled_answers)
        final_prediction = most_common

    is_hallucinating = semantic_score < HALLUCINATION_THRESHOLD
    is_accurate = (final_prediction == correct_option)

    # 2. Generate Medical Explanation and Medication Info
    explanation_prompt = (
        f"Medical Question: {question}\n"
        f"The selected answer is: {final_prediction}) {options.get(final_prediction, 'Unknown')}\n"
        "Briefly explain why this is the diagnosis/answer, and state if any specific medication "
        "or treatment is required."
    )
    explanation = get_llm_response(explanation_prompt, temp=0.1, max_tokens=250)

    return {
        "question": question,
        "ground_truth": correct_option,
        "predicted": final_prediction,
        "is_accurate": is_accurate,
        "semantic_score": round(semantic_score, 2),
        "is_hallucinating": is_hallucinating,
        "explanation_and_medication": explanation
    }



In [16]:
def main():
    # 1. Check which model is loaded before doing anything else
    active_model = get_loaded_model()
    if not active_model:
        print("Exiting script. Please load a model in LM Studio and ensure the server is running.")
        return

    print("Loading openlifescienceai/medmcqa dataset...")
    # Loading the validation split for testing
    dataset = load_dataset("openlifescienceai/medmcqa", split="validation")
    
    results = []
    correct_count = 0
    hallucination_count = 0

    print(f"Starting evaluation for {NUM_QUESTIONS} questions...")
    
    for i in range(NUM_QUESTIONS):
        print(f"\nProcessing question {i + 1}/{NUM_QUESTIONS}...")
        item = dataset[i]
        
        row_data = evaluate_question(item)
        results.append(row_data)
        
        # Update metrics
        if row_data["is_accurate"]: correct_count += 1
        if row_data["is_hallucinating"]: hallucination_count += 1
        
        # Save every 5 questions
        if (i + 1) % SAVE_INTERVAL == 0:
            df = pd.DataFrame(results)
            df.to_csv(CSV_FILENAME, index=False)
            print(f"--> Checkpoint saved at question {i + 1} to {CSV_FILENAME}")

    # Final calculations
    accuracy_pct = (correct_count / NUM_QUESTIONS) * 100
    hallucination_pct = (hallucination_count / NUM_QUESTIONS) * 100

    print("\n" + "="*40)
    print("FINAL RESULTS")
    print("="*40)
    print(f"Total Questions Evaluated: {NUM_QUESTIONS}")
    print(f"Overall Accuracy: {accuracy_pct:.2f}%")
    print(f"Overall Hallucination Rate: {hallucination_pct:.2f}%")
    
    # Append the summary metrics to the bottom of the CSV
    summary_data = {
        "question": "SUMMARY_METRICS",
        "ground_truth": "",
        "predicted": "",
        "is_accurate": f"Accuracy: {accuracy_pct:.2f}%",
        "semantic_score": "",
        "is_hallucinating": f"Hallucination Rate: {hallucination_pct:.2f}%",
        "explanation_and_medication": ""
    }
    df = pd.DataFrame(results + [summary_data])
    df.to_csv(CSV_FILENAME, index=False)
    print(f"Final results and metrics saved to {CSV_FILENAME}")

if __name__ == "__main__":
    main()

Checking connection to LM Studio and loaded model...
SUCCESS: Currently loaded model is -> llama3-med42-8b

Loading openlifescienceai/medmcqa dataset...
Starting evaluation for 400 questions...

Processing question 1/400...

Processing question 2/400...
--> Checkpoint saved at question 2 to offline_llm_medmcqa_results_LLama_med42.csv

Processing question 3/400...

Processing question 4/400...
--> Checkpoint saved at question 4 to offline_llm_medmcqa_results_LLama_med42.csv

Processing question 5/400...

Processing question 6/400...
--> Checkpoint saved at question 6 to offline_llm_medmcqa_results_LLama_med42.csv

Processing question 7/400...

Processing question 8/400...
--> Checkpoint saved at question 8 to offline_llm_medmcqa_results_LLama_med42.csv

Processing question 9/400...

Processing question 10/400...
--> Checkpoint saved at question 10 to offline_llm_medmcqa_results_LLama_med42.csv

Processing question 11/400...

Processing question 12/400...
--> Checkpoint saved at question